# HTML Tag Autocomplete (Character-Level LSTM)

A tiny character-level LSTM that learns a few HTML form tags such as `<input type='text'>`. Given the start of a tag it predicts the next characters one at a time until it closes the tag with `>`. It is a toy example of how code autocompletion works.

**How to run:** Open in Google Colab and choose **Runtime > Run all** (a CPU runtime is fine).

_Rewritten from my original Colab notebook: `Copy of gess the html tag.ipynb`._

## 1. Imports

In [ ]:
import numpy as np
import keras
from keras import layers

keras.utils.set_random_seed(42)

## 2. Dataset
Every tag ends with `>` so the model can learn where a tag stops.

In [ ]:
html_tags = [
    "<form>", "<input type='text'>", "<input type='password'>", "<input type='submit'>",
    "<input type='checkbox'>", "<input type='radio'>", "<input type='email'>", "<input type='button'>",
    "<label>", "<button>", "<select>", "<option>", "<textarea>",
]

PAD = "_"  # padding character, id 0
chars = [PAD] + sorted(set("".join(html_tags)))
char_to_id = {c: i for i, c in enumerate(chars)}
id_to_char = {i: c for c, i in char_to_id.items()}
MAX_LEN = max(len(t) for t in html_tags)
print("Characters:", "".join(chars[1:]))

## 3. Build (prefix -> next character) samples

In [ ]:
X_list, y_list = [], []
for tag in html_tags:
    for i in range(1, len(tag)):
        X_list.append([char_to_id[c] for c in tag[:i]])
        y_list.append(char_to_id[tag[i]])

X = keras.utils.pad_sequences(X_list, maxlen=MAX_LEN, padding="pre", value=0)
y = np.array(y_list)
print("Samples:", X.shape)

## 4. Model and training

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(MAX_LEN,)),
    layers.Embedding(len(chars), 32, mask_zero=True),
    layers.LSTM(128),
    layers.Dense(len(chars), activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(X, y, epochs=200, batch_size=32, verbose=0)
print("Final training accuracy:", round(history.history["accuracy"][-1], 3))

## 5. Autocomplete

In [ ]:
def complete_tag(prefix, max_new_chars=30):
    text = prefix.lower()
    for _ in range(max_new_chars):
        ids = [char_to_id.get(c, 0) for c in text]
        x = keras.utils.pad_sequences([ids], maxlen=MAX_LEN, padding="pre", truncating="pre")
        probs = model.predict(x, verbose=0)[0]
        probs[0] = 0  # never output padding
        next_char = id_to_char[int(np.argmax(probs))]
        text += next_char
        if next_char == ">":
            break
    return text


for prefix in ["<f", "<input type='p", "<input type='ch", "<but"]:
    print(f"{prefix!r:20} -> {complete_tag(prefix)}")

In [ ]:
model.save("html_tag_autocomplete.keras")